## 파이프라인
- 데이터 전처리 및 모델링 코드를 깔끔하게 구성하는 간단한 방법
- 전처리와 모델링 단계를 하나로 묶어, 전체를 마치 하나의 작업처럼 사용할 수 있게 해준다.

### ✅ 파이프라인의 장점
- 더 깔끔한 코드 : 전처리 단계마다 데이터를 따로 관리하려면 코드가 복잡해진다. 파이프라인을 쓰면 각 단계를 직접 관리하지 않아도 된다.
- 버그 감소 : 전처리 단계 누락이나 잘못된 적용을 줄일 수 있다.
- 배포가 쉬움 : 프로토타입에서 실제 서비스로 전활할 때 파이프라인은 많은 도움이 된다.
- 검증 방식 확장 가능 : 교차검증(cross-validation) 에서 특히 유용

---

### [Step 1] : 전처리 단계 정의

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 로드
melbourne_data = pd.read_csv('data/melb_data.csv')
y = melbourne_data.Price


melbourne_features = ['Type','Method','Regionname','Rooms','Distance','Postcode','Bedroom2', 'Bathroom', 'Landsize', 'Lattitude', 'Longtitude','Propertycount']
X = melbourne_data[melbourne_features]


# 훈련 데이터와 검증 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8, test_size=0.2,
                                                      random_state=0)

# 숫자형 데이터 전처리
#- 누락된 값을 완성하기 위한 단변량 입력기
# - 각 열에 기술 통계 (ex. 평균, 중앙값 또는 가장 빈번한 값)를 사용하거나 상수 값을 사용하여 누락된 값을 바꾼다.
num_transformer = SimpleImputer(strategy='constant')

# 범주형 데이터 전처리 (결측값 대체 + 원-핫 인코딩)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

X_train.info()

# 범주형 데이터 뽑아내기
categorical_cols = [ col for col in X_train.columns if X_train[col].dtype=='object']
numberical_cols = [col for col in X_train.columns if X_train[col].dtype!='object']

print(">>>",categorical_cols)
print("<<<", numberical_cols)

# 숫자형 + 범주형 전처리 묶기
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numberical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

<class 'pandas.core.frame.DataFrame'>
Index: 14716 entries, 2573 to 2732
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Type           14716 non-null  object 
 1   Method         14716 non-null  object 
 2   Regionname     14715 non-null  object 
 3   Rooms          14716 non-null  int64  
 4   Distance       14715 non-null  float64
 5   Postcode       14715 non-null  float64
 6   Bedroom2       11937 non-null  float64
 7   Bathroom       11936 non-null  float64
 8   Landsize       10887 non-null  float64
 9   Lattitude      12041 non-null  float64
 10  Longtitude     12041 non-null  float64
 11  Propertycount  14715 non-null  float64
dtypes: float64(8), int64(1), object(3)
memory usage: 1.5+ MB
>>> ['Type', 'Method', 'Regionname']
<<< ['Rooms', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Landsize', 'Lattitude', 'Longtitude', 'Propertycount']


### [Step 2] : 모델 정의

In [20]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=0)

### [Step 3] : 파이프라인 생성 및 평가

In [24]:
from sklearn.metrics import mean_absolute_error

# 파이프 라인 정의 (전처리 + 모델)
my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# 학습 : 전처리 + 모델 훈련
my_pipeline.fit(X_train, y_train)

# 예측 : 검증 데이터 전처리 + 예측
preds = my_pipeline.predict(X_valid)

# 평가
score = mean_absolute_error(y_valid, preds)
print('MAE: ', score)


MAE:  180079.81128748335


### 결론
- 파이프라인은 머신러닝 코드의 복잡성을 줄이고, 오류를 방지하는 데 유용
- 특히 복잡한 전처리가 필요한 워크플로우에서 매우 효과적